In [1]:
#Importar librerías
import requests
import pandas as pd

In [2]:
#Ciudades a consultar
ciudades = [
    {"nombre": "Bogotá",       "lat": 4.6097,  "lon": -74.0817, "region": "Andina"},
    {"nombre": "Medellín",     "lat": 6.2476,  "lon": -75.5658, "region": "Andina"},
    {"nombre": "Cali",         "lat": 3.4516,  "lon": -76.5320, "region": "Pacífica"},
    {"nombre": "Barranquilla", "lat": 10.9685, "lon": -74.7813, "region": "Caribe"},
    {"nombre": "Cartagena",    "lat": 10.3910, "lon": -75.4794, "region": "Caribe"},
    {"nombre": "Bucaramanga",  "lat": 7.1254,  "lon": -73.1198, "region": "Andina"},
]

In [3]:
#Consultar API para cada ciudad y guardar los resultados
url = "https://archive-api.open-meteo.com/v1/archive"

todos_los_datos = []

for ciudad in ciudades:
    print(f"Consultando {ciudad['nombre']}...")

    respuesta = requests.get(url, params={
        "latitude": ciudad["lat"],
        "longitude": ciudad["lon"],
        "start_date": "2024-01-01",
        "end_date": "2024-12-31",
        "daily": ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
                  "precipitation_sum", "windspeed_10m_max"],
        "timezone": "America/Bogota"
    })

    datos = respuesta.json()

    #Convertir a DataFrame
    df_ciudad = pd.DataFrame(datos["daily"])
    df_ciudad.columns = ["fecha", "temp_max", "temp_min", "temp_promedio", "lluvia_mm", "viento_kmh"]

    #Agregar info de la ciudad
    df_ciudad["ciudad"] = ciudad["nombre"]
    df_ciudad["region"] = ciudad["region"]

    todos_los_datos.append(df_ciudad)
    print(f"  OK - {len(df_ciudad)} registros")

#Unir todo en un DataFrame
df = pd.concat(todos_los_datos, ignore_index=True)
print(f"\nTotal de filas: {len(df)}")
df.head()

Consultando Bogotá...
  OK - 366 registros
Consultando Medellín...
  OK - 366 registros
Consultando Cali...
  OK - 366 registros
Consultando Barranquilla...
  OK - 366 registros
Consultando Cartagena...
  OK - 366 registros
Consultando Bucaramanga...
  OK - 366 registros

Total de filas: 2196


,fecha,temp_max,temp_min,temp_promedio,lluvia_mm,viento_kmh,ciudad,region
0,2024-01-01,20.1,9.5,14.2,1.1,8.7,Bogotá,Andina
1,2024-01-02,20.9,10.4,14.6,0.0,9.0,Bogotá,Andina
2,2024-01-03,20.9,8.6,14.3,0.0,7.5,Bogotá,Andina
3,2024-01-04,20.9,9.3,14.2,0.5,11.2,Bogotá,Andina
4,2024-01-05,21.0,7.4,14.0,0.0,7.5,Bogotá,Andina


In [4]:
#Ver el DataFrame
print(df.info())
print("\nValores nulos:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2196 entries, 0 to 2195
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   fecha          2196 non-null   object 
 1   temp_max       2196 non-null   float64
 2   temp_min       2196 non-null   float64
 3   temp_promedio  2196 non-null   float64
 4   lluvia_mm      2196 non-null   float64
 5   viento_kmh     2196 non-null   float64
 6   ciudad         2196 non-null   object 
 7   region         2196 non-null   object 
dtypes: float64(5), object(3)
memory usage: 137.4+ KB
None

Valores nulos:
fecha            0
temp_max         0
temp_min         0
temp_promedio    0
lluvia_mm        0
viento_kmh       0
ciudad           0
region           0
dtype: int64


In [5]:
#Convertir fecha y rellenar nulos
df["fecha"] = pd.to_datetime(df["fecha"])
df["lluvia_mm"] = df["lluvia_mm"].fillna(0)
df["temp_promedio"] = df["temp_promedio"].fillna(df["temp_promedio"].mean())

print("Tipos de datos después de limpiar:")
print(df.dtypes)

Tipos de datos después de limpiar:
fecha            datetime64[ns]
temp_max                float64
temp_min                float64
temp_promedio           float64
lluvia_mm               float64
viento_kmh              float64
ciudad                   object
region                   object
dtype: object


In [6]:
#Agregar columnas útiles para el análisis
df["mes"] = df["fecha"].dt.month
df["mes_nombre"] = df["fecha"].dt.strftime("%B")
df["trimestre"] = "Q" + df["fecha"].dt.quarter.astype(str)

#Categoría de lluvia
def tipo_lluvia(mm):
    if mm == 0:
        return "Sin lluvia"
    elif mm < 10:
        return "Lluvia leve"
    elif mm < 30:
        return "Lluvia moderada"
    else:
        return "Lluvia fuerte"

df["categoria_lluvia"] = df["lluvia_mm"].apply(tipo_lluvia)

#Categoría de temperatura
def tipo_temp(t):
    if t < 15:
        return "Fría"
    elif t < 22:
        return "Templada"
    elif t < 28:
        return "Cálida"
    else:
        return "Muy cálida"

df["categoria_temp"] = df["temp_promedio"].apply(tipo_temp)

print("Columnas del dataset:")
print(df.columns.tolist())
df.head()

Columnas del dataset:
['fecha', 'temp_max', 'temp_min', 'temp_promedio', 'lluvia_mm', 'viento_kmh', 'ciudad', 'region', 'mes', 'mes_nombre', 'trimestre', 'categoria_lluvia', 'categoria_temp']


,fecha,temp_max,temp_min,temp_promedio,lluvia_mm,viento_kmh,ciudad,region,mes,mes_nombre,trimestre,categoria_lluvia,categoria_temp
0,2024-01-01,20.1,9.5,14.2,1.1,8.7,Bogotá,Andina,1,January,Q1,Lluvia leve,Fría
1,2024-01-02,20.9,10.4,14.6,0.0,9.0,Bogotá,Andina,1,January,Q1,Sin lluvia,Fría
2,2024-01-03,20.9,8.6,14.3,0.0,7.5,Bogotá,Andina,1,January,Q1,Sin lluvia,Fría
3,2024-01-04,20.9,9.3,14.2,0.5,11.2,Bogotá,Andina,1,January,Q1,Lluvia leve,Fría
4,2024-01-05,21.0,7.4,14.0,0.0,7.5,Bogotá,Andina,1,January,Q1,Sin lluvia,Fría


In [7]:
#Resumen por ciudad
df.groupby("ciudad")[["temp_promedio", "lluvia_mm"]].mean().round(2)

,temp_promedio,lluvia_mm
ciudad,,
Barranquilla,27.91,4.14
Bogotá,14.20,2.34
Bucaramanga,22.19,9.94
Cali,22.49,6.58
Cartagena,28.02,4.11
Medellín,20.80,5.95


In [8]:
# Exportar CSV
df.to_csv("clima_colombia_2024.csv", index=False, encoding="utf-8-sig")
print("Archivo guardado como clima_colombia_2024.csv")
print(f"Filas: {len(df)} | Columnas: {len(df.columns)}")

Archivo guardado como clima_colombia_2024.csv
Filas: 2196 | Columnas: 13
